# Evaluating Recorded Coding Agent Traces

Coding agents — Claude Code, Cursor, OpenCode — expose lifecycle hooks. TruLens'
client-hook runtime listens to them and writes each turn to your TruLens
database as an ordinary record: `RECORD_ROOT` → `AGENT` → `GENERATION` + `TOOL`
spans, carrying OTel `gen_ai.*` attributes.

This notebook assumes that has **already happened**. It starts from a TruLens
database that already contains coding-agent traces, and is only about the
evaluation half: what is in there, what can be selected, and which metrics are
worth running.

The trace is where coding agents get interesting. A per-answer score says almost
nothing about an agent that ran eleven tools to answer one question; what you
want to know is whether it picked sensible tools, whether it wasted work, and
whether the turn even completed. So most metrics here are **trace-level**.

Companion notebook: [`genai_semconv_attribute_selection.ipynb`](./genai_semconv_attribute_selection.ipynb)
covers the selector mechanics against `gen_ai.*` attributes in general.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/truera/trulens/blob/main/examples/expositional/otel_genai/coding_agent_trace_evaluation.ipynb)

## Where the data came from

For reference only — this is the recording step, already done before this
notebook starts:

```bash
pip install trulens-apps-claude   # or trulens-apps-cursor, trulens-apps-opencode

export TRULENS_DESTINATION=local
export TRULENS_DATABASE_URL=sqlite:///$HOME/.trulens/client-hooks.sqlite

# Content is metadata-only by default; opt in to capture prompts and payloads.
export TRULENS_CAPTURE_CONTENT=true
export TRULENS_CAPTURE_TOOL_PAYLOADS=true

trulens-client-hooks install claude --project
```

Then the agent is used normally. Hook subprocesses append to a durable journal
and a detached worker drains it, so turns survive crashes and restarts.
`TRULENS_DESTINATION` also accepts `snowflake` and `otlp`. None of that matters
below — metrics select attributes, not storage.

## Point at the database

In [1]:
# !pip install trulens trulens-providers-cortex snowflake-snowpark-python

In [2]:
import collections
import json
import os

# The run lifecycle is a Snowflake-only feature; quiet it for local SQLite.
os.environ["TRULENS_MANAGE_RUNS"] = "false"

import pandas as pd
from snowflake.snowpark import Session
from trulens.core import Metric
from trulens.core import TruSession
from trulens.core.database.connector.default import DefaultDBConnector
from trulens.core.feedback.selector import Selector
from trulens.core.feedback.selector import Trace
from trulens.otel.semconv.trace import GenAIAttributes
from trulens.otel.semconv.trace import SpanAttributes
from trulens.providers.cortex import Cortex

Package jsonschema not present in requirements.


`~/.trulens/client-hooks.sqlite` is the default destination the client hooks
write to. Override `CODING_AGENT_DB` to point somewhere else — including a
Snowflake or Postgres URL, since nothing below depends on the backend.

One thing to be deliberate about: **computing metrics writes back to this
database.** Evaluation results are stored as `EVAL` spans next to the records
they score, which is what lets the dashboard show them together. If you would
rather not touch your live hook database, copy it first and point at the copy —
that is what the outputs below were produced against.

In [3]:
CODING_AGENT_DB = os.environ.get(
    "CODING_AGENT_DB", os.path.expanduser("~/.trulens/client-hooks.sqlite")
)
print("reading:", CODING_AGENT_DB)

session = TruSession(
    connector=DefaultDBConnector(database_url=f"sqlite:///{CODING_AGENT_DB}")
)

# Hook databases written by an older TruLens may predate the current schema.
session.migrate_database()

reading: /tmp/genai_verify/claude_run2.sqlite
Database schema is behind the expected revision. Please upgrade it by running `TruSession().migrate_database()` or reset it by running `TruSession().reset_database()`.
🦑 Migrating DB ...
DB Migration complete!
DB Validation complete!


## What is in here?

Coding-agent records use the client as the app name and the native client
version as the app version, so the app list tells you which agents and which
releases are present.

In [4]:
apps = pd.DataFrame([
    {
        "app_name": app.get("app_name"),
        "app_version": app.get("app_version"),
        "app_id": app.get("app_id"),
    }
    for app in session.get_apps()
]).drop_duplicates().reset_index(drop=True)

apps

,app_name,app_version,app_id
0,claude,2.1.19,app_hash_283ee61ac22f9d563c4abab9d1022c2e
1,claude,unknown,app_hash_85e97ffbc991219d87c84754bf031a03
2,claude,3.17.19,app_hash_70ebfdcc13f2a5079500d2bb80d928ed


Pick one agent and version to evaluate. `get_events` returns the raw spans for
that scope, which is what every metric below is computed from.

In [5]:
APP_NAME = os.environ.get("CODING_AGENT_APP", "claude")
APP_VERSION = os.environ.get("CODING_AGENT_VERSION", "2.1.19")

events = session.get_events(app_name=APP_NAME, app_version=APP_VERSION)
print(f"{APP_NAME} {APP_VERSION}: {len(events)} spans")


def attributes_of(row) -> dict:
    attrs = row["record_attributes"]
    return attrs if isinstance(attrs, dict) else json.loads(attrs)


span_types = collections.Counter(
    attributes_of(row).get(SpanAttributes.SPAN_TYPE) for _, row in events.iterrows()
)
dict(span_types)

claude 2.1.19: 62 spans


{'record_root': 11, 'agent': 14, 'generation': 11, 'tool': 26}

## What can be selected?

Before writing a metric, look at which attributes are actually present. This is
the fastest way to avoid a metric that silently produces nothing — and it is
necessary here, because content and tool payloads are only recorded if the
session opted in.

In [6]:
available = collections.defaultdict(collections.Counter)
for _, row in events.iterrows():
    attrs = attributes_of(row)
    span_type = attrs.get(SpanAttributes.SPAN_TYPE)
    for key in attrs:
        if key.startswith("gen_ai") or "coding_agent" in key:
            available[span_type][key] += 1

pd.DataFrame(
    [
        {"span_type": span_type, "attribute": key, "spans": count}
        for span_type, keys in available.items()
        for key, count in sorted(keys.items())
    ]
).sort_values(["span_type", "attribute"]).reset_index(drop=True)

,span_type,attribute,spans
0,agent,ai.observability.coding_agent.client,3
1,agent,ai.observability.coding_agent.native_event,3
2,agent,ai.observability.coding_agent.workspace,3
3,generation,gen_ai.operation.name,11
4,generation,gen_ai.request.model,11
5,generation,gen_ai.response.model,11
6,generation,gen_ai.system,7
7,generation,gen_ai.usage.input_tokens,7
8,generation,gen_ai.usage.output_tokens,7
9,tool,ai.observability.coding_agent.client,26


## Orient on the records

Read the traces before scoring them. Each record is one turn: the prompt, the
final response, and the ordered tool calls in between.

In [7]:
def summarize(events_df: pd.DataFrame) -> pd.DataFrame:
    per_record = {}
    for _, row in events_df.iterrows():
        attrs = attributes_of(row)
        record_id = attrs.get(SpanAttributes.RECORD_ID)
        entry = per_record.setdefault(
            record_id,
            {"conversation": None, "prompt": None, "response": None, "tools": []},
        )
        entry["conversation"] = entry["conversation"] or attrs.get(
            SpanAttributes.CONVERSATION_ID
        )
        span_type = attrs.get(SpanAttributes.SPAN_TYPE)
        if span_type == SpanAttributes.SpanType.TOOL.value:
            entry["tools"].append((
                row["start_timestamp"],
                attrs.get(GenAIAttributes.TOOL.NAME),
            ))
        elif span_type == SpanAttributes.SpanType.RECORD_ROOT.value:
            entry["prompt"] = str(attrs.get(SpanAttributes.RECORD_ROOT.INPUT))
            entry["response"] = str(attrs.get(SpanAttributes.RECORD_ROOT.OUTPUT))

    rows = []
    for entry in per_record.values():
        ordered = [name for _, name in sorted(entry["tools"], key=lambda t: t[0])]
        rows.append({
            "conversation": str(entry["conversation"])[:8],
            "prompt": (entry["prompt"] or "")[:52],
            "response": (entry["response"] or "")[:52],
            "n_tools": len(ordered),
            "tools": " ".join(ordered)[:46],
        })
    return pd.DataFrame(rows).sort_values(["conversation"]).reset_index(drop=True)


summarize(events)

,conversation,prompt,response,n_tools,tools
0,53cd6664,test,API Error: Invalid URL,0,
1,634510c6,quick test,Ready to help. What do you need?,0,
2,8e80531e,what's cool about sqlite,SQLite has several qualities that make it stan...,0,
3,8e80531e,cool - nice that it's the trulens default I',"Yes, SQLite is a solid default for TruLens - i...",0,
4,8e80531e,is the trulens path easy enough for more scale...,"Yes, the path is quite smooth. Here's what I f...",11,Task Glob Grep Read Read Glob Read Bash Read R
5,a349b93f,how does feedback.py work?,"API Error: 401 {""type"":""error"",""error"":{""type""...",0,
6,abd3f752,explain feedback.py,API Error: Invalid URL,0,
7,abd3f752,"cat > ~/.claude/settings.json <<EOF\n{\n ""env...",API Error: Invalid URL,0,
8,ef029d88,is trulens enterprise scale?,"Based on my research of the codebase, **yes, T...",5,Task Grep Read Read Read
9,f221a1b1,expexplain feedback.py,## Overview of `feedback.py`\n\nThe `feedback....,3,Glob Read Read


Two features of this real data drive the metric choices below.

Some turns did not complete at all — the response is a client-side
`API Error: ...` string. Notably, **nothing marks these as failed**: there is no
`error.type` and no `record_root.error`, because from the hook's point of view
the turn stopped normally. A metric has to read the output to notice.

Other turns are substantial research tasks with up to eleven tool calls, mostly
`Read`, `Grep`, `Glob` and `Task`. Those are the ones where trace-level judgement
earns its keep.

## Metrics

### Record-level: did the turn actually complete?

The cheapest useful metric on real agent data. Selects the root output and looks
for a client error, catching failures that carry no error attribute.

In [8]:
CLIENT_ERROR_MARKERS = (
    "api error",
    "authentication_error",
    "invalid bearer token",
    "invalid url",
    "rate limit",
    "please run /login",
)


def turn_completed(response: str) -> float:
    text = (response or "").strip().lower()
    if not text:
        return 0.0
    # Client errors are short and lead with the failure.
    return 0.0 if any(m in text[:200] for m in CLIENT_ERROR_MARKERS) else 1.0


m_completed = Metric(
    implementation=turn_completed,
    name="Turn Completed",
).on({"response": Selector.select_record_output()})

### Per-tool-call: shell command guardrail

A policy check on every `TOOL` span, reading the official `gen_ai.tool.name` and
`gen_ai.tool.call.arguments`. Non-shell tools pass trivially; shell commands are
screened for destructive operations. No LLM, fully auditable.

In [9]:
DESTRUCTIVE = ("rm -rf", "git reset --hard", "git push --force", "drop table")
SHELL_TOOLS = {"Bash", "bash", "Shell", "shell"}


def shell_command_is_safe(call: dict) -> float:
    if call["name"] not in SHELL_TOOLS:
        return 1.0
    arguments = call["arguments"] or "{}"
    try:
        command = json.loads(arguments).get("command", "")
    except (TypeError, ValueError):
        command = str(arguments)
    return 0.0 if any(bad in command.lower() for bad in DESTRUCTIVE) else 1.0


m_command_safety = Metric(
    implementation=shell_command_is_safe,
    name="Shell Command Safety",
).on({
    "call": Selector(
        span_type=SpanAttributes.SpanType.TOOL,
        span_attributes_processor=lambda attrs: {
            "name": attrs.get(GenAIAttributes.TOOL.NAME),
            "arguments": attrs.get(GenAIAttributes.TOOL.CALL_ARGUMENTS),
        },
    )
})

### Trace-level: wasted work

This is the class of metric that a single-span selector cannot express, and the
reason `trace_level=True` exists. Repeating an identical tool call — the same
file read twice, the same search run twice — is measurable waste, but only
visible across spans.

`Trace.events` is a DataFrame of every span in the record, so sorting by
`start_timestamp` recovers the agent's actual sequence of actions.

In [10]:
def ordered_tool_calls(trace: Trace) -> list:
    """Tool spans in execution order: (name, arguments)."""
    calls = []
    for _, event in trace.events.iterrows():
        attrs = event["record_attributes"]
        if not isinstance(attrs, dict):
            attrs = json.loads(attrs)
        if attrs.get(SpanAttributes.SPAN_TYPE) != SpanAttributes.SpanType.TOOL.value:
            continue
        calls.append((
            event["start_timestamp"],
            attrs.get(GenAIAttributes.TOOL.NAME),
            attrs.get(GenAIAttributes.TOOL.CALL_ARGUMENTS) or "{}",
        ))
    calls.sort(key=lambda call: call[0])
    return [(name, arguments) for _, name, arguments in calls]


def no_redundant_tool_calls(trace: Trace) -> float:
    """Fraction of tool calls that were not exact repeats."""
    calls = ordered_tool_calls(trace)
    if not calls:
        return 1.0  # nothing to repeat
    unique = len(set(calls))
    return unique / len(calls)


m_no_redundancy = Metric(
    implementation=no_redundant_tool_calls,
    name="No Redundant Tool Calls",
).on({"trace": Selector(trace_level=True)})

A note on the variant you probably want if your agent *writes* code rather than
just reading it: the same `ordered_tool_calls` helper supports a
"did it verify its own work" check — find the last `Edit` or `Write`, then assert
some later `Bash` call ran the tests. The sessions in this database are read-only
research tasks, so it would pass vacuously here and is left out.

### Trace-level: built-in agentic judges

TruLens ships LLM judges that take the whole trace, serialise it (with
compression, so long traces stay in budget), and score it against a rubric.
`Tool Selection` asks whether the chosen tools were appropriate for the request;
`Execution Efficiency` asks whether the agent reached the goal without wasted
work.

Also available on any `LLMProvider`: `plan_adherence_with_cot_reasons`,
`plan_quality_with_cot_reasons`, `tool_calling_with_cot_reasons`,
`tool_quality_with_cot_reasons`, `logical_consistency_with_cot_reasons`.

In [11]:
snowpark_session = Session.builder.config(
    "connection_name", os.environ.get("SNOWFLAKE_CONNECTION_NAME", "default")
).create()
provider = Cortex(snowpark_session=snowpark_session, model_engine="claude-sonnet-4-5")

m_tool_selection = Metric(
    implementation=provider.tool_selection_with_cot_reasons,
    name="Tool Selection",
).on({"trace": Selector(trace_level=True)})

m_efficiency = Metric(
    implementation=provider.execution_efficiency_with_cot_reasons,
    name="Execution Efficiency",
).on({"trace": Selector(trace_level=True)})

### Conversation-level

The hooks wrote the native session id as `ai.observability.conversation_id`, so
`.on_conversation()` groups the turns of one coding session in order and scores
them together. The result attaches to the session's last turn.

In [12]:
m_conversation = Metric(
    implementation=provider.conversation_helpfulness_with_cot_reasons,
    name="Conversation Helpfulness",
).on_conversation()

## Compute

These records were written by the hook runtime, not recorded through a `TruApp`,
so there is no app object holding a metric list. `compute_feedbacks_on_events`
evaluates a metric list against an events DataFrame — the entry point for any
already-recorded trace, whatever produced it.

In [13]:
session.compute_feedbacks_on_events(
    events,
    [
        m_completed,
        m_command_safety,
        m_no_redundancy,
        m_tool_selection,
        m_efficiency,
        m_conversation,
    ],
)
session.force_flush()

True

In [14]:
records, metric_names = session.get_records_and_feedback(
    app_name=APP_NAME, app_version=APP_VERSION
)

present = [name for name in metric_names if name in records.columns]
report = records[["input"] + present].copy()
report["input"] = report["input"].str[:44]
report.round(3)

,input,Turn Completed,No Redundant Tool Calls,Tool Selection,Execution Efficiency,Conversation Helpfulness,Shell Command Safety
0,quick test,1.0,1.0,1.000,1.000,0.333,NaN
1,what's cool about sqlite,1.0,1.0,1.000,0.333,NaN,NaN
2,cool - nice that it's the trulens default I,1.0,1.0,1.000,1.000,NaN,NaN
3,is the trulens path easy enough for more sca,1.0,1.0,1.000,0.333,1.000,1.0
4,is trulens enterprise scale?,1.0,1.0,0.667,0.333,1.000,1.0
5,how does feedback.py work?,0.0,1.0,0.000,1.000,0.000,NaN
6,explain feedback.py,0.0,1.0,0.000,0.333,NaN,NaN
7,"cat > ~/.claude/settings.json <<EOF\n{\n ""env",0.0,1.0,0.000,0.333,0.000,NaN
8,test,0.0,1.0,0.000,0.000,0.000,NaN
9,expexplain feedback.py,1.0,1.0,0.667,0.333,1.000,1.0


## What the scores say

**`Turn Completed` is the most valuable metric here, and the cheapest.** It
scores 0.0 on exactly the four turns whose response is a client `API Error`.
Nothing on any span marked those as failures — no `error.type`, no
`record_root.error` — so reading the root output was the only way to find them.

**Gate the expensive judges on it.** Look at the `how does feedback.py work?`
row: the turn died on a 401, ran no tools, and still scored 1.0 on
`Execution Efficiency`. A trace judge handed a near-empty trace has nothing to
assess and its score is not meaningful. Conversely the low `Tool Selection`
scores on the failed turns reflect an absent trace, not a misbehaving agent.
Filter to completed turns before drawing conclusions from either judge.

**`No Redundant Tool Calls` is 1.0 across the board** — there were no
byte-identical repeats, even in the eleven-call turn. That is worth contrasting
with `Execution Efficiency`, which penalised several of those same turns. The two
do not agree, and the reason is instructive: exact matching only catches literal
repetition, while the judge also counts *near*-duplicates and general wandering —
reading three files where one would have done. Treat the deterministic metric as
a cheap floor that catches an obvious pathology, not as a substitute for the
judge.

**`Shell Command Safety` is `NaN` wherever a record has no tool spans**, and 1.0
where it does. These sessions were read-heavy, with one `Bash` call among them.
That is the expected shape of a guardrail: mostly silent, valuable on the day it
is not.

**`Conversation Helpfulness` has one value per session**, on that session's last
turn. The 0.0 scores belong to sessions that consisted only of failed turns.

## Dashboard

Metrics were written back as `EVAL` spans next to their records, so the dashboard
shows scores alongside the trace tree — the tool calls, their arguments, and
their results.

In [ ]:
from trulens.dashboard import run_dashboard

run_dashboard(session)

## Takeaways

- A recorded coding-agent session is an ordinary TruLens record. Evaluating one
  needs no special API: point a `TruSession` at the database the hooks wrote to,
  select attributes, and call `compute_feedbacks_on_events`. No `TruApp`, and no
  re-running the agent.
- Check what is actually present before writing metrics. Content and tool
  payloads are opt-in at record time, so `gen_ai.tool.call.arguments` may simply
  not be there — and a metric selecting it would quietly score nothing.
- The highest-value metric on real data was the cheapest one. Agent turns fail in
  ways that set no error attribute, so a completion check that reads the root
  output catches failures no span-level signal exposes.
- Gate LLM trace judges on completion. On a degenerate or empty trace their
  scores are noise, in both directions.
- The metrics specific to agents are trace-level, because they are about
  *sequence*. `Trace.events` sorted by `start_timestamp` is the whole API.
- Deterministic trace metrics and trace judges answer overlapping but different
  questions. Exact-match redundancy is a floor; the judge sees waste that exact
  matching cannot.